<a href="https://colab.research.google.com/github/aslamrosul/Algoritma_Struktur_Data_1G_06/blob/main/Modul7PCVK.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================
# PRAKTIKUM MODUL 7
# Filter Spasial: Low Pass, High Pass, Point Detection, Line Detection, Edge Detection
# ============================================

import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt
import math
from google.colab import drive
from google.colab.patches import cv2_imshow

# --------------------------------------------
# 1️⃣ MOUNT GOOGLE DRIVE
# --------------------------------------------
drive.mount('/content/drive')

# --------------------------------------------
# 2️⃣ FUNGSI KONVOLUSI MANUAL
# --------------------------------------------

def konvolusi_manual(citra, kernel):
    """Melakukan konvolusi 2D manual antara citra grayscale dan kernel."""
    m, n = kernel.shape
    if len(citra.shape) == 3:
        citra = cv.cvtColor(citra, cv.COLOR_BGR2GRAY)

    pad = m // 2
    citra_padded = np.pad(citra, ((pad, pad), (pad, pad)), mode='constant', constant_values=0)
    hasil = np.zeros_like(citra)

    for i in range(pad, citra_padded.shape[0] - pad):
        for j in range(pad, citra_padded.shape[1] - pad):
            region = citra_padded[i - pad:i + pad + 1, j - pad:j + pad + 1]
            hasil[i - pad, j - pad] = np.sum(region * kernel)

    hasil = np.clip(hasil, 0, 255)
    return hasil.astype(np.uint8)

# --------------------------------------------
# 3️⃣ LOAD CITRA
# --------------------------------------------
path = '/content/drive/MyDrive/Polinema/Kuliah/PCVK/Images/lena.jpg'
img = cv.imread(path, cv.IMREAD_GRAYSCALE)
cv2_imshow(img)
print("Ukuran citra:", img.shape)

# --------------------------------------------
# 4️⃣ DEFINISI KERNEL
# --------------------------------------------

kernel_dict = {
    "average": np.ones((3,3))/9,
    "low_pass": np.array([[1,1,1],[1,4,1],[1,1,1]])/12,
    "high_pass": np.array([[-1,0,1],[-1,0,3],[-3,0,1]]),
    "sharpen": np.array([[0,-1,0],[-1,5,-1],[0,-1,0]]),
    "emboss": np.array([[-2,-1,0],[-1,1,1],[0,1,2]]),
    "sobel_left": np.array([[1,0,-1],[2,0,-2],[1,0,-1]]),
    "prewitt": np.array([[-1,0,1],[-1,0,1],[-1,0,1]]),
    "canny": np.array([[-1,-1,-1],[-1,8,-1],[-1,-1,-1]])
}

# --------------------------------------------
# 5️⃣ APLIKASIKAN FILTER
# --------------------------------------------

fig, axes = plt.subplots(3,3, figsize=(12,12))
axes = axes.ravel()

for idx, (nama, kernel) in enumerate(kernel_dict.items()):
    hasil = konvolusi_manual(img, kernel)
    axes[idx].imshow(hasil, cmap='gray')
    axes[idx].set_title(nama)
    axes[idx].axis('off')

plt.tight_layout()
plt.show()

# --------------------------------------------
# 6️⃣ FILTER GAUSSIAN (5x5 dan 21x21)
# --------------------------------------------

def gaussian_kernel(size):
    sigma = math.sqrt(size)
    gk = cv.getGaussianKernel(size, sigma)
    kernel = gk @ gk.T
    return kernel / np.sum(kernel)

gauss5 = gaussian_kernel(5)
gauss21 = gaussian_kernel(21)

blur5 = konvolusi_manual(img, gauss5)
blur21 = konvolusi_manual(img, gauss21)

plt.figure(figsize=(12,6))
plt.subplot(1,3,1); plt.imshow(img, cmap='gray'); plt.title('Original')
plt.subplot(1,3,2); plt.imshow(blur5, cmap='gray'); plt.title('Gaussian Blur 5x5')
plt.subplot(1,3,3); plt.imshow(blur21, cmap='gray'); plt.title('Gaussian Blur 21x21')
plt.show()

# --------------------------------------------
# 7️⃣ PERBANDINGAN FILTER
# --------------------------------------------

filters = ['average', 'low_pass', 'high_pass', 'sharpen', 'emboss', 'sobel_left', 'prewitt', 'canny']
plt.figure(figsize=(16,10))
for i, f in enumerate(filters):
    hasil = konvolusi_manual(img, kernel_dict[f])
    plt.subplot(2,4,i+1)
    plt.imshow(hasil, cmap='gray')
    plt.title(f.capitalize())
    plt.axis('off')
plt.tight_layout()
plt.show()
